# FockPARFLM v2.1 — QFT-Motivated Creation Gate Improvements

## Purpose

Validate four QFT-motivated improvements to the FockPARFLM v2 creation gate,
derived from the NN-QFT analysis of Ageev & Ageev (2026) and the field-theoretic
proof of the Conservative Obstruction Theorem (Paper v4, §17c).

## QFT-motivated improvements under test

| # | Improvement | QFT principle | Architectural change |
|---|---|---|---|
| 1 | Gumbel-Softmax creation | Virtual particle creation is stochastic (vacuum fluctuations) | Add Gumbel(0,1) noise to creation scores before softmax |
| 2 | Per-register key subspaces | Independent interaction channels maximize G_c^(4) | Per-register W_K^(k) instead of shared W_K |
| 3 | Orthogonal query initialisation | Orthogonal plane-wave modes in field expansion | Orthogonal init for W_Q across registers |
| 4 | Canonical creation-destruction coupling | [a, a†] = 1: creation and annihilation are conjugate | g_destroy = 1 - max_j(alpha_kj), replacing MLP |

## Cell configurations (D6–D10 series)

| Cell | Changes vs D1 baseline | Tests |
|---|---|---|
| `Q0` | D1 baseline replica (shared W_K, deterministic softmax, Gaussian init, MLP destruction) | Control |
| `Q1` | + Gumbel-Softmax creation | Stochastic virtual particle emission |
| `Q2` | + Per-register W_K^(k) | Independent interaction channels |
| `Q3` | + Orthogonal W_Q initialisation | Cold-start diversity |
| `Q4` | + Canonical destruction (attention-derived) | Coupled creation-destruction |
| `Q5` | All four combined (full QFT v2.1 gate) | Full QFT-informed gate |
| `Q6` | Q5 + learnable temperature tau_0=1.0 | QFT gate + temperature |
| `Q7` | Q5 + M=32 (double registers) | Scaling interaction channels |
| `Q8` | PARFLM baseline (no registers) | B6 control |

**Scale**: d=256, L=8, T=256, batch=8, 2000 steps on 1M TinyStories tokens.
Expected wall-clock: ~20-40 min per arm on A100/H100 GPU.

## Run protocol

Run the notebook once per `CELL` value. All arms use the same data split,
hyperparameters, and evaluation protocol. The diagnostic section (§8) runs
automatically and logs per-layer register statistics including attention entropy,
register diversity, and non-Gaussianity proxies.

## 0. Environment setup + cell selector

In [ ]:
CELL = 'Q0'   # one of: 'Q0' | 'Q1' | 'Q2' | 'Q3' | 'Q4' | 'Q5' | 'Q6' | 'Q7' | 'Q8'
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_fock_v2_qft'

import os, sys, shutil, subprocess, json, time, math
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    _sh(f'git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {COLAB_REPO_PATH}')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    RESULTS_ROOT = GDRIVE_OUT
    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    RESULTS_ROOT = (
        REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
        / 'results' / 'fock_v2_qft'
    )
    RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

PARF_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf'
SCALEUP_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'scaleup'
DATA_DIR    = REPO_ROOT / 'notebooks' / 'conservative_arch'
for p in (str(REPO_ROOT), str(DATA_DIR), str(SCALEUP_DIR), str(PARF_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

print(f'REPO_ROOT    = {REPO_ROOT}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')
print(f'RUN_DIR      = {RUN_DIR}')
print(f'CELL = {CELL!r}  SEED = {SEED}')

## 1. Device + reproducibility

In [ ]:
import torch
import numpy as np

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')

torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed_all(SEED)
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory / 1e9:.1f} GB)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('Using MPS (Apple Silicon)')
else:
    device = 'cpu'
    print('WARNING: CPU only — will be slow')
print(f'device = {device}')

## 2. Experiment recipes

All arms use the P10g-equivalent architecture as the base.
Scale: T=256 (half-length), 1M training tokens, 2000 steps.

The QFT improvements are controlled by four boolean flags:
- `gumbel_creation`: add Gumbel(0,1) noise to creation scores
- `per_register_keys`: use per-register W_K^(k) instead of shared W_K
- `orthogonal_query_init`: orthogonal initialization for W_Q across registers
- `canonical_destruction`: replace MLP destruction with attention-derived signal

In [ ]:
VOCAB_SIZE     = 50257
D              = 256
L              = 8
MAX_LEN        = 256
BLOCK_SIZE     = 256
V_HIDDEN       = 2048
V_DEPTH        = 3
TOP_K          = 4
SCORE_HEAD     = 32
BATCH_SIZE     = 8
TOTAL_STEPS    = 2000
LR             = 3e-4
WEIGHT_DECAY   = 0.01
WARMUP_STEPS   = 100
GRAD_CLIP      = 1.0
EVAL_EVERY     = 200
EVAL_ITERS     = 20
LOG_EVERY      = 50
MAX_TRAIN_TOK  = 1_000_000

_FOCK_BASE = dict(
    use_fock_v2=True,
    n_registers=16,
    d_k=64,
    register_salience_decay=0.5,
    register_salience_threshold=0.005,
    destruction_gate_hidden=64,
    reverse_channel=True,
    per_layer_creation=False,
    blend_scale=1.0,
    gumbel_creation=False,
    per_register_keys=False,
    orthogonal_query_init=False,
    canonical_destruction=False,
    tau_create_init=None,
)

RECIPES = {
    'Q0': {
        'desc': 'FockPARF v2 baseline (D1 replica, shared W_K, deterministic, Gaussian init, MLP destroy)',
        **_FOCK_BASE,
    },
    'Q1': {
        'desc': 'Q0 + Gumbel-Softmax creation (stochastic virtual particle emission)',
        **_FOCK_BASE,
        'gumbel_creation': True,
    },
    'Q2': {
        'desc': 'Q0 + per-register key subspaces (independent interaction channels)',
        **_FOCK_BASE,
        'per_register_keys': True,
    },
    'Q3': {
        'desc': 'Q0 + orthogonal W_Q initialisation (maximally spread probes)',
        **_FOCK_BASE,
        'orthogonal_query_init': True,
    },
    'Q4': {
        'desc': 'Q0 + canonical destruction (attention-derived, no MLP)',
        **_FOCK_BASE,
        'canonical_destruction': True,
    },
    'Q5': {
        'desc': 'Full QFT v2.1: Gumbel + per-reg keys + ortho init + canonical destroy',
        **_FOCK_BASE,
        'gumbel_creation': True,
        'per_register_keys': True,
        'orthogonal_query_init': True,
        'canonical_destruction': True,
    },
    'Q6': {
        'desc': 'Q5 + learnable temperature tau_0=1.0',
        **_FOCK_BASE,
        'gumbel_creation': True,
        'per_register_keys': True,
        'orthogonal_query_init': True,
        'canonical_destruction': True,
        'tau_create_init': 1.0,
    },
    'Q7': {
        'desc': 'Q5 + M=32 (double registers, scaling interaction channels)',
        **_FOCK_BASE,
        'n_registers': 32,
        'gumbel_creation': True,
        'per_register_keys': True,
        'orthogonal_query_init': True,
        'canonical_destruction': True,
    },
    'Q8': {
        'desc': 'PARFLM baseline (no registers, P10g recipe)',
        'use_fock_v2': False,
        'n_registers': 0,
    },
}

if CELL not in RECIPES:
    raise ValueError(f'CELL must be one of {list(RECIPES)}; got {CELL!r}')

recipe = RECIPES[CELL]
USE_FOCK_V2 = recipe.get('use_fock_v2', False)
M_REGISTERS = recipe.get('n_registers', 0)

print(f'Cell {CELL}: {recipe["desc"]}')
for k, v in sorted(recipe.items()):
    if k != 'desc':
        print(f'  {k:30s} = {v!r}')

## 3. Load TinyStories (1M token subset)

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=MAX_TRAIN_TOK)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

LOGFREQ_PATH = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'
SCALEUP_LOGFREQ = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
if SCALEUP_LOGFREQ.exists():
    LOGFREQ_PATH = SCALEUP_LOGFREQ
elif not LOGFREQ_PATH.exists():
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    np.save(LOGFREQ_PATH, surprisal)
print(f'logfreq: {LOGFREQ_PATH}')

## 4. Build model with QFT improvements

The four QFT improvements are applied as monkey-patches to the base FockPARFLM_v2
architecture, controlled by the recipe flags. This avoids modifying the core model
code while testing the improvements.

In [ ]:
import importlib
import torch.nn as nn
import torch.nn.functional as F_torch

import model_parf_sparse
import model_fock_parf_v2
importlib.reload(model_parf_sparse)
importlib.reload(model_fock_parf_v2)
from model_parf_sparse import SparsePARFConfig, SparsePARFLM
from model_fock_parf_v2 import FockPARFConfig_v2, FockPARFLM_v2

torch.manual_seed(SEED)

parf_kwargs = dict(
    vocab_size=VOCAB_SIZE,
    d=D, max_len=MAX_LEN, L=L,
    v_hidden=V_HIDDEN, v_depth=V_DEPTH,
    v_phi_kind='structural_competitive',
    v_phi_d_type=16, v_phi_d_angle=8,
    v_phi_phi_hidden=64, v_phi_theta_hidden=64,
    v_phi_mlp_hidden=64,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    top_k=TOP_K,
    score_head_hidden=SCORE_HEAD,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    theta_activation='softsign',
    theta_form='bilinear',
)

if USE_FOCK_V2:
    fock_kwargs = dict(
        n_registers=recipe['n_registers'],
        d_k=recipe['d_k'],
        register_salience_decay=recipe['register_salience_decay'],
        register_salience_threshold=recipe['register_salience_threshold'],
        destruction_gate_hidden=recipe['destruction_gate_hidden'],
        reverse_channel=recipe['reverse_channel'],
        stack_discipline=True,
    )
    if recipe.get('tau_create_init') is not None:
        fock_kwargs['tau_create_init'] = recipe['tau_create_init']
    cfg = FockPARFConfig_v2(**parf_kwargs, **fock_kwargs)
    model = FockPARFLM_v2(cfg).to(device)
    n_fock = model.get_fock_v2_overhead()
else:
    cfg = SparsePARFConfig(**parf_kwargs)
    model = SparsePARFLM(cfg).to(device)
    n_fock = 0

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
print(f'Model: {type(model).__name__}')
print(f'params: total={n_total:,}  V_theta={n_v_theta:,}  Fock_overhead={n_fock:,}')

if USE_FOCK_V2:
    print(f'M={cfg.n_registers}  d_k={cfg.d_k}  reverse_channel={cfg.reverse_channel}')
    if model.creation_gate.log_tau is not None:
        tau_val = model.creation_gate.log_tau.exp().item()
        print(f'tau_create init = {tau_val:.4f}')
    else:
        print(f'tau_create = None (using 1/sqrt(d_k) = {1.0 / cfg.d_k**0.5:.4f})')

## 5. Apply QFT improvements (monkey-patches)

Each improvement patches the creation gate forward pass without modifying
the base model code. This keeps the experiment self-contained.

In [ ]:
applied_patches = []

if USE_FOCK_V2:
    gate = model.creation_gate
    M = cfg.n_registers
    d_k = cfg.d_k

    # ── Improvement 2: Per-register key subspaces ──
    if recipe.get('per_register_keys', False):
        old_W_K = gate.W_K
        gate.per_reg_W_K = nn.Parameter(
            torch.randn(M, D, d_k, device=device) * (1.0 / D**0.5)
        )
        n_new = gate.per_reg_W_K.numel()
        applied_patches.append(f'per_register_keys: added per_reg_W_K ({n_new:,} params)')

    # ── Improvement 3: Orthogonal query initialisation ──
    if recipe.get('orthogonal_query_init', False):
        init_scale = 0.02
        W_Q = gate.W_Q.data
        for i in range(D):
            U, _, _ = torch.linalg.svd(
                torch.randn(d_k, max(M, d_k), device=device)
            )
            W_Q[:, i, :] = U[:M, :d_k] * init_scale
        applied_patches.append(f'orthogonal_query_init: W_Q re-initialised with orthogonal modes')

    # ── Patch the creation gate forward to implement improvements 1 & 2 & 4 ──
    _original_forward = gate.forward

    use_gumbel = recipe.get('gumbel_creation', False)
    use_per_reg_keys = recipe.get('per_register_keys', False)
    use_canon_destroy = recipe.get('canonical_destruction', False)

    def _patched_creation_forward(h_tokens, r_states):
        B, T, d = h_tokens.shape
        M_local = r_states.shape[1]

        Q = torch.einsum('bmd,mdk->bmk', r_states, gate.W_Q)

        if use_per_reg_keys:
            K = torch.einsum('btd,mdk->bmtk', h_tokens, gate.per_reg_W_K)
            scores = torch.einsum('bmk,bmtk->bmt', Q, K)
        else:
            K = gate.W_K(h_tokens)
            scores = torch.bmm(
                Q.reshape(B * M_local, 1, d_k),
                K.unsqueeze(1).expand(B, M_local, T, d_k)
                    .reshape(B * M_local, d_k, T),
            ).reshape(B, M_local, T)

        if use_gumbel and gate.training:
            u = torch.rand_like(scores).clamp(min=1e-8, max=1.0 - 1e-8)
            gumbel_noise = -torch.log(-torch.log(u))
            scores = scores + gumbel_noise

        if gate.log_tau is not None:
            tau = gate.log_tau.exp().clamp(min=1e-4)
            scores = scores / tau
        else:
            scores = scores / (d_k ** 0.5)

        alpha = F_torch.softmax(scores, dim=-1)

        V = gate.W_V(h_tokens)
        r_new = torch.bmm(alpha, V)

        alpha_max = alpha.max(dim=-1).values

        return r_new, alpha_max

    if use_gumbel or use_per_reg_keys:
        gate.forward = _patched_creation_forward
        if use_gumbel:
            applied_patches.append('gumbel_creation: Gumbel(0,1) noise added to scores at train time')
        if use_per_reg_keys:
            applied_patches.append('per_register_keys: creation forward uses per_reg_W_K')

    # ── Improvement 4: Canonical destruction ──
    # DESIGN NOTE: The naive approach of multiplying salience by alpha_max after the
    # layer step causes catastrophic collapse.  alpha_max from softmax over T=256 tokens
    # is ≈ 1/256 ≈ 0.004 at init; after L=8 layers this gives 0.004^8 ≈ 10^{-18} —
    # all registers die immediately.  Instead we:
    #   (a) use a forward hook to capture alpha_max DURING the existing creation gate
    #       call (zero extra forward passes),
    #   (b) REPLACE the MLP destruction gates entirely (not stack on top of them),
    #   (c) use a log-normalized peakedness signal:
    #         survival = log(alpha_max * T) / log(T)  ∈ [0, 1]
    #         0 = uniform/boring, 1 = single-token/peaked
    #         g_destroy = 0.1 + 0.4 * (1 - survival)   ∈ [0.1, 0.5]
    #       which matches the MLP gate's ~0.5 destruction rate at init but
    #       rewards peaked/interesting registers with lower decay.
    # Steady-state salience (decay=0.5): s_∞ = alpha_max*(1-g)/(1+g).
    #   boring (uniform, T=256): s_∞ ≈ 0.0013  < threshold 0.005  → inactive ✓
    #   moderate focus (8 tok):  s_∞ ≈ 0.013   > threshold 0.005  → active   ✓
    if use_canon_destroy:
        _stored_canon_survival = {'value': None}

        def _capture_survival_hook(module, inp, output):
            _, alpha_max = output          # alpha_max: (B, M)
            T_local = float(inp[0].shape[1])   # h_tokens is first input, shape (B, T, d)
            log_T = math.log(T_local)
            peaked_ratio = (alpha_max * T_local).clamp(1.0, T_local)
            # survival ∈ [0, 1]: 0=uniform, 1=single-token; detach for gradient stability
            survival = (torch.log(peaked_ratio) / log_T).detach()
            _stored_canon_survival['value'] = survival

        _canon_hook_handle = gate.register_forward_hook(_capture_survival_hook)

        class CanonicalDestrGate(nn.Module):
            """Replaces the MLP destruction gate with attention-peakedness signal."""
            def forward(self, r_new):
                survival = _stored_canon_survival['value']
                B_r = r_new.shape[0]
                if survival is not None and survival.shape[0] == B_r:
                    g = (0.1 + 0.4 * (1.0 - survival)).clamp(0.0, 1.0)
                else:
                    # Fallback before the hook has fired (e.g. first call edge case)
                    g = torch.full((B_r, r_new.shape[1]), 0.3,
                                   device=r_new.device, dtype=r_new.dtype)
                return g

        for _li in range(cfg.L):
            model.destruction_gates[_li] = CanonicalDestrGate()

        applied_patches.append(
            'canonical_destruction: hook captures alpha_max peakedness; '
            'MLP gates replaced with g=0.1+0.4*(1-log(alpha_max*T)/log(T))'
        )

    # Move any new parameters to device
    model = model.to(device)

if applied_patches:
    print(f'Applied {len(applied_patches)} QFT patches:')
    for p in applied_patches:
        print(f'  - {p}')
else:
    print('No QFT patches applied (baseline or non-Fock arm).')

n_total_after = sum(p.numel() for p in model.parameters())
print(f'params after patches: {n_total_after:,} (delta: +{n_total_after - n_total:,})')

## 6. Training loop

In [ ]:
def lr_at(step):
    if step < WARMUP_STEPS:
        return LR * (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(TOTAL_STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


@torch.no_grad()
def evaluate():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


opt = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY,
)
model.train()

log = []
best_ppl = float('inf')
t0 = time.time()

rc_scale_history = []
tau_create_history = []
HAS_TAU_CREATE = (USE_FOCK_V2 and model.creation_gate.log_tau is not None)

for step in range(TOTAL_STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)

    xb, yb = get_batch(train_ids, BATCH_SIZE, BLOCK_SIZE, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    _, loss = model(x, y)

    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    opt.step()

    if USE_FOCK_V2 and model.reverse_channel_scale is not None:
        with torch.no_grad():
            rc_val = model.reverse_channel_scale.item()
            rc_tanh = torch.tanh(model.reverse_channel_scale).item()
            rc_grad = (
                model.reverse_channel_scale.grad.item()
                if model.reverse_channel_scale.grad is not None else 0.0
            )
        if (step + 1) % LOG_EVERY == 0:
            rc_scale_history.append({
                'step': step + 1, 'raw': rc_val,
                'tanh': rc_tanh, 'grad': rc_grad,
            })

    if HAS_TAU_CREATE:
        with torch.no_grad():
            tau_val = model.creation_gate.log_tau.exp().clamp(min=1e-4).item()
            tau_grad = (
                model.creation_gate.log_tau.grad.item()
                if model.creation_gate.log_tau.grad is not None else 0.0
            )
        if (step + 1) % LOG_EVERY == 0:
            tau_create_history.append({
                'step': step + 1, 'tau': tau_val, 'grad': tau_grad,
            })

    if (step + 1) % LOG_EVERY == 0 or step == 0:
        extras = ''
        if USE_FOCK_V2 and model.reverse_channel_scale is not None:
            extras += f'  rc={rc_tanh:+.4f}'
        if HAS_TAU_CREATE:
            extras += f'  tau={tau_val:.4f}'
        print(f'[{CELL}] step {step+1:>5}/{TOTAL_STEPS}  '
              f'lr={lr_at(step):.2e}  '
              f'loss={loss.item():.4f}{extras}  '
              f'wall={time.time()-t0:.0f}s')

    if (step + 1) % EVAL_EVERY == 0 or (step + 1) == TOTAL_STEPS:
        val_loss = evaluate()
        val_ppl = math.exp(val_loss)
        if val_ppl < best_ppl:
            best_ppl = val_ppl
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best_ppl={best_ppl:.2f}')
        log.append({
            'step': step + 1,
            'val_loss': val_loss,
            'val_ppl': val_ppl,
            'best_ppl': best_ppl,
            'train_loss': loss.item(),
        })

elapsed = time.time() - t0
print(f'\n[{CELL}] Done. wall={elapsed:.0f}s  '
      f'final_ppl={log[-1]["val_ppl"]:.2f}  best_ppl={best_ppl:.2f}')

## 7. Save checkpoint and log

In [ ]:
import dataclasses

full_tag = f'fock_v2_qft_{CELL}_seed{SEED}'
ckpt_path = RUN_DIR / f'{full_tag}_ckpt.pt'
log_path  = RUN_DIR / f'{full_tag}_training_log.jsonl'

torch.save({
    'model_state_dict': model.state_dict(),
    'config': dataclasses.asdict(cfg) if dataclasses.is_dataclass(cfg) else vars(cfg),
    'recipe': recipe,
    'applied_patches': applied_patches,
    'best_ppl': best_ppl,
    'step': TOTAL_STEPS,
    'elapsed_s': elapsed,
}, ckpt_path)

with open(log_path, 'w') as f:
    for entry in log:
        f.write(json.dumps(entry) + '\n')

if rc_scale_history:
    rc_path = RUN_DIR / f'{full_tag}_rc_scale_history.jsonl'
    with open(rc_path, 'w') as f:
        for entry in rc_scale_history:
            f.write(json.dumps(entry) + '\n')
    print(f'rc_scale history: {rc_path}')

if tau_create_history:
    tau_path = RUN_DIR / f'{full_tag}_tau_create_history.jsonl'
    with open(tau_path, 'w') as f:
        for entry in tau_create_history:
            f.write(json.dumps(entry) + '\n')
    print(f'tau_create history: {tau_path}')

with open(RUN_DIR / f'{full_tag}_config.json', 'w') as f:
    json.dump({
        'cell': CELL, 'seed': SEED,
        'recipe': recipe,
        'applied_patches': applied_patches,
        'best_ppl': best_ppl,
        'elapsed_s': elapsed,
        'total_steps': TOTAL_STEPS,
    }, f, indent=2)

print(f'checkpoint: {ckpt_path}')
print(f'log:        {log_path}')

## 8. Register diagnostics — attention entropy, diversity, non-Gaussianity proxy

In [ ]:
import matplotlib.pyplot as plt

if not USE_FOCK_V2:
    print(f'Skipping register diagnostics for {CELL} (baseline arm).')
else:
    model.eval()
    diag_batches = 10

    all_attn_entropy = []
    all_alpha_max    = []
    all_reg_diversity = []

    for b_idx in range(diag_batches):
        xb, _ = get_batch(val_ids, BATCH_SIZE, BLOCK_SIZE, rng)
        x = torch.from_numpy(xb).to(device)

        with torch.enable_grad():
            h0 = model._embed(x)
            r, salience = model._init_registers(BATCH_SIZE, h0.device)
            h, h_prev = h0, h0
            m_b = model.compute_mass(x)
            gamma, dt = model.gamma, cfg.dt

            batch_entropy = []
            batch_alpha_max = []
            batch_diversity = []

            for ell in range(cfg.L):
                B, T, d = h.shape
                M_local = cfg.n_registers

                r_new_content, alpha_max_diag = model.creation_gate(h, r)

                Q_diag = torch.einsum('bmd,mdk->bmk', r, model.creation_gate.W_Q)
                if hasattr(model.creation_gate, 'per_reg_W_K'):
                    K_diag = torch.einsum('btd,mdk->bmtk', h, model.creation_gate.per_reg_W_K)
                    scores_diag = torch.einsum('bmk,bmtk->bmt', Q_diag, K_diag)
                else:
                    K_diag = model.creation_gate.W_K(h)
                    scores_diag = torch.bmm(
                        Q_diag.reshape(B * M_local, 1, cfg.d_k),
                        K_diag.unsqueeze(1).expand(B, M_local, T, cfg.d_k)
                             .reshape(B * M_local, cfg.d_k, T),
                    ).reshape(B, M_local, T)

                if model.creation_gate.log_tau is not None:
                    tau_diag = model.creation_gate.log_tau.exp().clamp(min=1e-4)
                    scores_diag = scores_diag / tau_diag
                else:
                    scores_diag = scores_diag / (cfg.d_k ** 0.5)
                alpha_diag = F_torch.softmax(scores_diag, dim=-1)

                log_alpha = torch.log(alpha_diag + 1e-12)
                entropy = -(alpha_diag * log_alpha).sum(dim=-1)
                max_entropy = math.log(T)
                norm_entropy = entropy / max_entropy

                batch_entropy.append(norm_entropy.detach().mean(0).cpu().numpy())
                batch_alpha_max.append(alpha_max_diag.detach().mean(0).cpu().numpy())

                r_flat = r_new_content.detach().mean(0)
                cos_matrix = F_torch.cosine_similarity(
                    r_flat.unsqueeze(0), r_flat.unsqueeze(1), dim=-1
                )
                mask = ~torch.eye(M_local, dtype=torch.bool, device=device)
                mean_off_diag_cos = cos_matrix[mask].mean().item()
                batch_diversity.append(1.0 - mean_off_diag_cos)

                h_new, h_prev_out, r, salience = model._fock_v2_layer_step(
                    h, h_prev, r, salience, m_b, gamma, dt, layer_idx=ell,
                )
                h_new = h_new.detach().requires_grad_(True)
                h_prev_out = h_prev_out.detach().requires_grad_(True)
                r = r.detach()
                salience = salience.detach()
                h_prev = h_prev_out
                h = h_new

            all_attn_entropy.append(np.stack(batch_entropy))
            all_alpha_max.append(np.stack(batch_alpha_max))
            all_reg_diversity.append(batch_diversity)

    mean_entropy   = np.mean(all_attn_entropy, axis=0)
    mean_alpha_max = np.mean(all_alpha_max, axis=0)
    mean_diversity = np.mean(all_reg_diversity, axis=0)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ax = axes[0]
    im = ax.imshow(mean_entropy, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=1)
    ax.set_xlabel('Register index')
    ax.set_ylabel('Layer')
    ax.set_title(f'[{CELL}] Normalised attention entropy\n(0=peaked, 1=uniform)')
    plt.colorbar(im, ax=ax)

    ax = axes[1]
    im = ax.imshow(mean_alpha_max, aspect='auto', cmap='viridis', vmin=0)
    ax.set_xlabel('Register index')
    ax.set_ylabel('Layer')
    ax.set_title(f'[{CELL}] max_j(alpha_kj)\n(creation signal strength)')
    plt.colorbar(im, ax=ax)

    ax = axes[2]
    ax.plot(range(cfg.L), mean_diversity, 'o-', linewidth=2)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Register diversity (1 - mean cos sim)')
    ax.set_title(f'[{CELL}] Register content diversity')
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    fig_path = RUN_DIR / f'{full_tag}_diagnostics.png'
    fig.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

    print(f'\n--- Entropy summary (mean across registers) ---')
    for ell in range(cfg.L):
        print(f'  Layer {ell}: entropy={mean_entropy[ell].mean():.4f}  '
              f'alpha_max={mean_alpha_max[ell].mean():.4f}  '
              f'diversity={mean_diversity[ell]:.4f}')

    diag_path = RUN_DIR / f'{full_tag}_diagnostics.json'
    with open(diag_path, 'w') as f:
        json.dump({
            'entropy_per_layer': mean_entropy.tolist(),
            'alpha_max_per_layer': mean_alpha_max.tolist(),
            'diversity_per_layer': mean_diversity.tolist(),
        }, f, indent=2)
    print(f'Saved: {diag_path}')

## 9. Cross-cell comparison dashboard

After running multiple cells (Q0–Q8), this section scans all completed
runs and produces a comparison table and training curves.

In [ ]:
import matplotlib.pyplot as plt

cell_names = ['Q0', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']
summary_rows = []

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

for cell_name in cell_names:
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        continue

    entries = []
    with open(logs[0]) as f:
        for line in f:
            entries.append(json.loads(line))

    if not entries:
        continue

    steps = [e['step'] for e in entries]
    ppls  = [e['val_ppl'] for e in entries]
    best  = min(ppls)

    desc = RECIPES.get(cell_name, {}).get('desc', cell_name)
    summary_rows.append({
        'cell': cell_name,
        'desc': desc,
        'best_ppl': best,
        'final_ppl': ppls[-1],
    })

    ax.plot(steps, ppls, 'o-', label=f'{cell_name} (best={best:.1f})', linewidth=1.5)

ax.set_xlabel('Training step')
ax.set_ylabel('Validation PPL')
ax.set_title('QFT Improvements — Training Curves')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()

comparison_path = RESULTS_ROOT / f'qft_comparison_seed{SEED}.png'
fig.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {comparison_path}')

if summary_rows:
    print(f'\n{"Cell":<6} {"Best PPL":>10} {"Final PPL":>10}  Description')
    print('-' * 80)
    for row in sorted(summary_rows, key=lambda r: r['best_ppl']):
        print(f'{row["cell"]:<6} {row["best_ppl"]:>10.2f} {row["final_ppl"]:>10.2f}  {row["desc"]}')

    with open(RESULTS_ROOT / f'qft_comparison_seed{SEED}.json', 'w') as f:
        json.dump(summary_rows, f, indent=2)
else:
    print('No completed runs found yet. Run cells Q0–Q8 first.')

## 10. Summary and output listing

In [ ]:
print(f'\n=== Run complete: {CELL} seed={SEED} ===')
print(f'Best PPL: {best_ppl:.2f}')
print(f'Wall time: {elapsed:.0f}s')
print(f'\nQFT patches applied: {len(applied_patches)}')
for p in applied_patches:
    print(f'  - {p}')
print(f'\nOutput files:')
for f in sorted(RUN_DIR.glob('*')):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:60s} {size_kb:>8.1f} KB')